# Phase 5: multi-role elite count

## In how many roles does each player reach the elite range, and where does Messi fall?

This is the project's central question, operationalized. Phase 4 showed
the four roles don't behave as one axis, so counting elite appearances
across them is meaningful, not redundant. Rather than commit to a single
arbitrary "elite" cutoff before seeing results, this notebook tests the
multi-role elite count across several percentile thresholds and reports
whether the finding holds across all of them or depends on which one is
chosen.

Any hard cutoff creates an artificial cliff: a 94.3rd-percentile score
would read as categorically "not elite" right next to a 95th-percentile
score reading as "elite" under a 95 threshold, even though the difference
is close to meaningless. So the elite/not-elite count is never reported
alone here, every table showing a count also shows the exact percentile
score per role, so a near-miss reads as a near-miss, not as absence.
Missing a threshold doesn't disqualify a player from the interpretation,
it's a way to operationalize something that's ultimately continuous, and
conclusions get drawn from the actual pattern of numbers, not mechanically
from whether a line was crossed.

The count is computed for the entire qualified population (n=2488), not
just for Messi, so his position can be read against the full distribution
of how many roles other players reach elite status in, including
reference players (Cristiano Ronaldo, Neymar, Mbappé, Ronaldinho) for
context.

---

# Fase 5: conteo de élite multi-rol

## ¿En cuántos roles llega cada jugador al rango de élite, y dónde cae Messi?

Esta es la pregunta central del proyecto, operacionalizada. La Fase 4
mostró que los cuatro roles no se comportan como un solo eje, así que
contar apariciones en el rango de élite entre ellos es significativo, no
redundante. En vez de comprometerse con un único corte de "élite"
arbitrario antes de ver resultados, este notebook prueba el conteo de
élite multi-rol en varios umbrales de percentil y reporta si el hallazgo
se sostiene en todos o depende de cuál se elija.

Cualquier corte duro crea un precipicio artificial: un puntaje en el
percentil 94.3 se leería como categóricamente "no élite" justo al lado de
un percentil 95 leído como "élite" bajo un umbral de 95, aunque la
diferencia sea casi insignificante. Por eso el conteo de élite/no-élite
nunca se reporta solo acá, toda tabla que muestre un conteo también
muestra el percentil exacto por rol, para que un "casi-adentro" se lea
como eso, no como ausencia. No superar un umbral no descalifica a un
jugador de la interpretación, es una forma de operacionalizar algo que en
el fondo es continuo, y las conclusiones se sacan del patrón real de
números, no mecánicamente de si se cruzó una línea.

El conteo se calcula para toda la población calificada (n=2488), no solo
para Messi, para poder leer su posición contra la distribución completa
de en cuántos roles llegan a élite otros jugadores, incluyendo jugadores
de referencia (Cristiano Ronaldo, Neymar, Mbappé, Ronaldinho) como
contexto.

In [1]:
import pandas as pd

role_scores = pd.read_csv("../data/processed/role_scores.csv")
role_metrics_df = pd.read_csv("../data/processed/role_metrics.csv")

role_cols = ["finisher_score", "dribbler_score", "chance_creator_score", "play_organizer_score"]

print(role_scores[role_cols].describe())

reference_players = ["Messi", "Cristiano Ronaldo", "Neymar", "Mbappé", "Mbappe", "Ronaldinho"]
for player in reference_players:
    match = role_scores[role_scores["name"].str.contains(player, case=False, na=False, regex=False)]
    if len(match) > 0:
        print(f"\n{player}:")
        print(match[["name"] + role_cols].to_string(index=False))
    else:
        print(f"\n{player}: no encontrado (revisar nombre exacto)")

       finisher_score  dribbler_score  chance_creator_score  \
count     2488.000000     2488.000000           2488.000000   
mean        50.020096       50.020096             50.020096   
std         24.444119       25.155280             21.563557   
min         17.852358        4.642283             18.938907   
25%         28.339362       29.513331             31.109325   
50%         46.359191       49.450697             49.093650   
75%         70.838357       70.600549             66.024920   
max        100.000000       99.531083             99.746785   

       play_organizer_score  
count           2488.000000  
mean              50.020096  
std               26.572354  
min                0.281350  
25%               28.136053  
50%               49.511656  
75%               72.458802  
max               99.967846  

Messi:
        name  finisher_score  dribbler_score  chance_creator_score  play_organizer_score
Lionel Messi       97.186495       95.726152             92.13022

## An early flag before defining thresholds

Looking at the reference players before running the full population
count: Neymar's profile (98.12 / 96.13 / 92.95 / 78.75) is strikingly
close to Messi's (97.19 / 95.73 / 92.13 / 84.06) across all four roles.
Ronaldinho actually exceeds Messi specifically in play organizer (87.78
vs. 84.06), despite being notably weaker as a finisher (75.38). This is
noted here, before any elite threshold gets applied, so it can't be read
as cherry-picked after the fact: the data is allowed to complicate the
hypothesis, and this is exactly that kind of complication. Whether it
changes the shape of the central finding is what the rest of this
notebook checks.

---

## Una señal temprana antes de definir los umbrales

Mirando a los jugadores de referencia antes de correr el conteo completo
de la población: el perfil de Neymar (98.12 / 96.13 / 92.95 / 78.75) es
sorprendentemente cercano al de Messi (97.19 / 95.73 / 92.13 / 84.06) en
los cuatro roles. Ronaldinho de hecho supera a Messi específicamente en
organizador de juego (87.78 contra 84.06), a pesar de ser notablemente
más débil como finalizador (75.38). Esto queda anotado acá, antes de
aplicar cualquier umbral de élite, para que no se pueda leer como elegido
a conveniencia después de los hechos: a los datos se les permite
complicar la hipótesis, y esto es exactamente ese tipo de complicación.
Si cambia o no la forma del hallazgo central es lo que revisa el resto de
este notebook.

In [3]:
thresholds = [85, 90, 95, 99]

for t in thresholds:
    for role in role_cols:
        role_scores[f"{role}_elite_{t}"] = role_scores[role] >= t
    elite_cols = [f"{role}_elite_{t}" for role in role_cols]
    role_scores[f"n_elite_roles_{t}"] = role_scores[elite_cols].sum(axis=1)

for t in thresholds:
    print(f"\nUmbral top {100-t}% (percentil >= {t}):")
    print(role_scores[f"n_elite_roles_{t}"].value_counts().sort_index())


Umbral top 15% (percentil >= 85):
n_elite_roles_85
0    1692
1     603
2     160
3      30
4       3
Name: count, dtype: int64

Umbral top 10% (percentil >= 90):
n_elite_roles_90
0    1969
1     436
2      75
3       8
Name: count, dtype: int64

Umbral top 5% (percentil >= 95):
n_elite_roles_95
0    2263
1     209
2      15
3       1
Name: count, dtype: int64

Umbral top 1% (percentil >= 99):
n_elite_roles_99
0    2457
1      31
Name: count, dtype: int64


In [4]:
for t in thresholds:
    messi_row = role_scores[role_scores["name"].str.contains("Messi", case=False, na=False)]
    n_elite = messi_row[f"n_elite_roles_{t}"].values[0]
    print(f"Umbral top {100-t}%: Messi elite en {n_elite} de 4 roles")

print("\n--- Jugadores en el máximo de roles de élite alcanzado en cada umbral ---")
for t in thresholds:
    max_roles = role_scores[f"n_elite_roles_{t}"].max()
    top_players = role_scores[role_scores[f"n_elite_roles_{t}"] == max_roles]
    print(f"\nUmbral top {100-t}% — máximo alcanzado: {max_roles} de 4 roles ({len(top_players)} jugador(es))")
    print(top_players[["name"] + role_cols].to_string(index=False))

Umbral top 15%: Messi elite en 3 de 4 roles
Umbral top 10%: Messi elite en 3 de 4 roles
Umbral top 5%: Messi elite en 2 de 4 roles
Umbral top 1%: Messi elite en 0 de 4 roles

--- Jugadores en el máximo de roles de élite alcanzado en cada umbral ---

Umbral top 15% — máximo alcanzado: 4 de 4 roles (3 jugador(es))
           name  finisher_score  dribbler_score  chance_creator_score  play_organizer_score
Zinedine Zidane       86.870311       86.334405             88.975080             92.564309
  Ermindo Onega       91.599678       87.774652             89.935691             94.722669
   Johan Cruyff       89.449357       98.666935             98.689711             86.543408

Umbral top 10% — máximo alcanzado: 3 de 4 roles (8 jugador(es))
            name  finisher_score  dribbler_score  chance_creator_score  play_organizer_score
         Robinho       93.662915       92.403537             62.049839             90.434084
    Lionel Messi       97.186495       95.726152             92.130

In [5]:
top_4de4 = role_scores[role_scores["name"].isin(["Zinedine Zidane", "Ermindo Onega", "Johan Cruyff"])]
top_4de4_ids = top_4de4["id"].tolist()

context_cols = ["name", "world_cups_played", "matches_played", "minutesPlayed"]
comparison = role_metrics_df[role_metrics_df["id"].isin(top_4de4_ids + [role_metrics_df[role_metrics_df["name"].str.contains("Messi", case=False, na=False)]["id"].values[0]])]

print(comparison[context_cols].to_string(index=False))

           name  world_cups_played  matches_played  minutesPlayed
Zinedine Zidane                  3              12         1109.0
   Lionel Messi                  6              34         3054.0
  Ermindo Onega                  1               4          360.0
   Johan Cruyff                  1               7          630.0


In [6]:
elite_3plus_85 = role_scores[role_scores["n_elite_roles_85"] >= 3] if "n_elite_roles_85" in role_scores.columns else None

# Si no calculaste el umbral 85 como columna, lo recreamos rápido
role_scores["n_elite_roles_85"] = (role_scores[role_cols] >= 85).sum(axis=1)
elite_3plus_85 = role_scores[role_scores["n_elite_roles_85"] >= 3]

elite_3plus_85_with_minutes = elite_3plus_85.merge(
    role_metrics_df[["id", "minutesPlayed", "world_cups_played"]], on="id"
)

print(f"Jugadores con >= 3 roles de élite en umbral top 15% (percentil 85): {len(elite_3plus_85_with_minutes)}")
print(elite_3plus_85_with_minutes[["name", "n_elite_roles_85", "minutesPlayed", "world_cups_played"]].sort_values("n_elite_roles_85", ascending=False).to_string(index=False))

print(f"\nMinutos promedio de este grupo: {elite_3plus_85_with_minutes['minutesPlayed'].mean():.0f}")
print(f"Minutos promedio de TODA la población calificada: {role_metrics_df['minutesPlayed'].mean():.0f}")

correlation = role_scores.merge(role_metrics_df[["id", "minutesPlayed"]], on="id")[["n_elite_roles_85", "minutesPlayed"]].corr().iloc[0, 1]
print(f"\nCorrelación entre minutos de carrera y cantidad de roles de élite alcanzados: {correlation:.3f}")

Jugadores con >= 3 roles de élite en umbral top 15% (percentil 85): 33
                  name  n_elite_roles_85  minutesPlayed  world_cups_played
       Zinedine Zidane                 4         1109.0                  3
          Johan Cruyff                 4          630.0                  1
         Ermindo Onega                 4          360.0                  1
              Denílson                 3          375.0                  2
               Robinho                 3          494.0                  2
        Ángel Di María                 3         1304.0                  4
          Arjen Robben                 3         1347.0                  3
       James Rodríguez                 3          869.0                  3
            Ronaldinho                 3          766.0                  2
                Neymar                 3         1225.0                  4
Diego Armando Maradona                 3         1940.0                  4
         Michael Olise       

In [7]:
role_scores["role_score_range"] = role_scores[role_cols].max(axis=1) - role_scores[role_cols].min(axis=1)
role_scores["role_score_std"] = role_scores[role_cols].std(axis=1)

# Sensibilidad del umbral de minutos, transparente, sin reemplazar 270
minutes_thresholds = [270, 900, 1500]
for mt in minutes_thresholds:
    qualified_at_mt = role_metrics_df[role_metrics_df["minutesPlayed"] >= mt]["id"]
    subset = role_scores[role_scores["id"].isin(qualified_at_mt)]
    max_4de4 = subset[subset["n_elite_roles_85"] == 4]
    print(f"Minutos >= {mt}: {len(subset)} jugadores en la sub-población, "
          f"{len(max_4de4)} llegan a 4/4 (percentil 85): {max_4de4['name'].tolist()}")

print()
top_3plus = role_scores[role_scores["n_elite_roles_85"] >= 3].merge(
    role_metrics_df[["id", "minutesPlayed"]], on="id"
)
print("Perfil parejo vs. perfil de picos, entre quienes llegan a 3+ roles de élite:")
print(top_3plus[["name", "n_elite_roles_85", "role_score_range", "minutesPlayed"] + role_cols]
      .sort_values("role_score_range")
      .to_string(index=False))

Minutos >= 270: 2488 jugadores en la sub-población, 3 llegan a 4/4 (percentil 85): ['Zinedine Zidane', 'Ermindo Onega', 'Johan Cruyff']
Minutos >= 900: 258 jugadores en la sub-población, 1 llegan a 4/4 (percentil 85): ['Zinedine Zidane']
Minutos >= 1500: 43 jugadores en la sub-población, 0 llegan a 4/4 (percentil 85): []

Perfil parejo vs. perfil de picos, entre quienes llegan a 3+ roles de élite:
                  name  n_elite_roles_85  role_score_range  minutesPlayed  finisher_score  dribbler_score  chance_creator_score  play_organizer_score
       Zinedine Zidane                 4          6.229904         1109.0       86.870311       86.334405             88.975080             92.564309
         Ermindo Onega                 4          6.948017          360.0       91.599678       87.774652             89.935691             94.722669
          Johan Cruyff                 4         12.146302          630.0       89.449357       98.666935             98.689711             86.543408

In [8]:
elite_90_group = role_scores[role_scores["n_elite_roles_90"] == 3].merge(
    role_metrics_df[["id", "minutesPlayed", "world_cups_played", "matches_played"]], on="id"
)
print("Los 8 jugadores que llegan al techo real (3 de 4) en el umbral genuino de élite (top 10%):")
print(elite_90_group[["name", "minutesPlayed", "world_cups_played", "matches_played"] + role_cols]
      .sort_values("minutesPlayed", ascending=False)
      
      .to_string(index=False))

Los 8 jugadores que llegan al techo real (3 de 4) en el umbral genuino de élite (top 10%):
            name  minutesPlayed  world_cups_played  matches_played  finisher_score  dribbler_score  chance_creator_score  play_organizer_score
    Lionel Messi         3054.0                  6              34       97.186495       95.726152             92.130225             84.059486
          Neymar         1225.0                  4              15       98.124330       96.128081             92.954180             78.754019
       Rivellino         1206.0                  3              15       95.585477       95.016077             72.471865             93.818328
   Gheorghe Hagi         1039.0                  3              12       92.309753       97.025723             92.427653             82.612540
         Robinho          494.0                  2               8       93.662915       92.403537             62.049839             90.434084
     Heinz Flohe          454.0                  2 

## A second lens: average role score, not just elite-role count

The binary elite count treats "crosses the threshold in 3 roles by a wide
margin, misses the 4th by a little" the same as "barely crosses the
threshold in 4 roles, dominant in none." Those are different stories. A
simple average of the four percentile scores is a complementary way to
summarize overall level across all four roles without depending on any
particular cutoff at all, it doesn't replace the elite count, it answers
a related but distinct question: not "how many thresholds does this
player clear," but "how high is this player's overall multi-role
profile."

This also lets the minimum-minutes question get resolved with evidence
instead of intuition. The ranking is computed at the official 270-minute
threshold, and then re-checked at higher floors (900, 1500 minutes) to
see whether short-sample players near the top of the average ranking
survive a stricter sample requirement, or whether they were only there
because of how little football is needed to sustain a high rate over a
handful of matches.

---

## Una segunda mirada: promedio de puntaje de rol, no solo el conteo de élite

El conteo binario de élite trata igual a "cruza el umbral en 3 roles por
mucho margen, falla el 4to por poco" que a "apenas cruza el umbral en 4
roles, sin dominar en ninguno". Son historias distintas. Un promedio
simple de los cuatro puntajes de percentil es una forma complementaria de
resumir el nivel general en los cuatro roles sin depender de ningún corte
en particular, no reemplaza el conteo de élite, responde una pregunta
relacionada pero distinta: no "cuántos umbrales cruza este jugador", sino
"qué tan alto es el perfil multi-rol general de este jugador".

Esto también permite resolver la pregunta del mínimo de minutos con
evidencia en vez de intuición. El ranking se calcula con el umbral
oficial de 270 minutos, y después se revisa de nuevo con pisos más altos
(900, 1500 minutos) para ver si los jugadores de muestra corta cerca del
top del ranking de promedio sobreviven un requisito de muestra más
estricto, o si estaban ahí solo por cuánto poco fútbol hace falta para
sostener una tasa alta en un puñado de partidos.

In [10]:
role_scores["role_score_avg"] = role_scores[role_cols].mean(axis=1)
role_scores_with_minutes = role_scores.merge(role_metrics_df[["id", "minutesPlayed", "world_cups_played"]], on="id")

for mt in [270, 900, 1500]:
    subset = role_scores_with_minutes[role_scores_with_minutes["minutesPlayed"] >= mt]
    top20 = subset.sort_values("role_score_avg", ascending=False).head(20)
    messi_rank = subset.sort_values("role_score_avg", ascending=False).reset_index(drop=True)
    messi_position = messi_rank[messi_rank["name"].str.contains("Messi", case=False, na=False)].index[0] + 1

    print(f"\n=== Minutos >= {mt} ({len(subset)} jugadores) ===")
    print(f"Messi ocupa el puesto #{messi_position} por promedio de puntaje de rol")
    print(top20[["name", "role_score_avg", "minutesPlayed", "world_cups_played"] + role_cols].to_string(index=False))


=== Minutos >= 270 (2488 jugadores) ===
Messi ocupa el puesto #2 por promedio de puntaje de rol
                  name  role_score_avg  minutesPlayed  world_cups_played  finisher_score  dribbler_score  chance_creator_score  play_organizer_score
          Johan Cruyff       93.337353          630.0                  1       89.449357       98.666935             98.689711             86.543408
          Lionel Messi       92.275589         3054.0                  6       97.186495       95.726152             92.130225             84.059486
                Neymar       91.490153         1225.0                  4       98.124330       96.128081             92.954180             78.754019
         Gheorghe Hagi       91.093917         1039.0                  3       92.309753       97.025723             92.427653             82.612540
         Ermindo Onega       91.008173          360.0                  1       91.599678       87.774652             89.935691             94.722669
      Car

## Average role score across career-length samples: the strongest evidence yet, with an honest caveat

At the official 270-minute threshold, Messi is 2nd by average role score
(92.28), narrowly behind Johan Cruyff (93.34), whose 630 minutes
represent his entire World Cup career, a single tournament. The moment a
serious sample size is required, the picture is unambiguous: at 900+
minutes (258 players), Messi leads the entire population (92.28, ahead of
Neymar at 91.49 and Hagi at 91.09). At 1500+ minutes (43 players), Messi
still leads by a wide margin (92.28 vs. Maradona's 89.24 in 2nd, then a
steep drop to Enzo Scifo at 81.30).

This is a different, complementary claim to the elite-count result:
nobody in the population clears the individual per-role elite bar in all
four roles simultaneously at a genuine threshold (90th percentile or
higher), but by overall composite level averaged across all four
dimensions, restricted to players with a real career-length sample,
Messi's average is the highest in the dataset. Both are true at once and
both get reported, not one selected over the other.

A useful contrast: Cristiano Ronaldo, with a comparable career sample
(2206 minutes, also 6 World Cups), ranks 14th in the 1500+ group (69.52),
dragged down by a sharply peaked profile (96.6 as a finisher, but 61.1 and
37.1 as a creator and organizer). This is the concrete illustration of why
"peak in one dimension" and "sustained high level across four dimensions"
are different claims, and why the average-score lens matters alongside
the elite count.

---

## Promedio de puntaje de rol en muestras de carrera larga: la evidencia más fuerte hasta ahora, con una salvedad honesta

Con el umbral oficial de 270 minutos, Messi queda 2do por promedio de
puntaje de rol (92.28), apenas detrás de Johan Cruyff (93.34), cuyos 630
minutos representan toda su carrera mundialista, un solo torneo. En
cuanto se exige una muestra seria, el panorama es inequívoco: con 900+
minutos (258 jugadores), Messi lidera toda la población (92.28, por
delante de Neymar con 91.49 y Hagi con 91.09). Con 1500+ minutos (43
jugadores), Messi sigue liderando por un margen amplio (92.28 contra el
89.24 de Maradona en 2do lugar, y después una caída pronunciada hasta el
81.30 de Enzo Scifo).

Esta es una afirmación distinta y complementaria al resultado del conteo
de élite: nadie en la población supera el umbral de élite individual por
rol en los cuatro roles simultáneamente en un umbral genuino (percentil
90 o superior), pero por nivel compuesto general promediado en las
cuatro dimensiones, restringido a jugadores con una muestra real de
carrera, el promedio de Messi es el más alto del dataset. Las dos cosas
son ciertas a la vez y las dos se reportan, no se elige una por sobre la
otra.

Un contraste útil: Cristiano Ronaldo, con una muestra de carrera
comparable (2206 minutos, también 6 Mundiales), queda 14to en el grupo de
1500+ (69.52), arrastrado por un perfil muy puntiagudo (96.6 como
finalizador, pero 61.1 y 37.1 como creador y organizador). Esta es la
ilustración concreta de por qué "pico en una dimensión" y "nivel alto
sostenido en cuatro dimensiones" son afirmaciones distintas, y por qué la
mirada del promedio importa junto al conteo de élite.

In [11]:
reference_names = ["Lionel Messi", "Cristiano Ronaldo", "Neymar", "Kylian Mbappé", "Ronaldinho", "Diego Armando Maradona"]
ceiling_85_names = top_4de4["name"].tolist()  # Zidane, Onega, Cruyff
ceiling_90_names = elite_90_group["name"].tolist()  # los 8 del techo real

avg_top2_names = []
for mt in [270, 900, 1500]:
    subset = role_scores_with_minutes[role_scores_with_minutes["minutesPlayed"] >= mt]
    top2 = subset.sort_values("role_score_avg", ascending=False).head(2)["name"].tolist()
    avg_top2_names.extend(top2)

synthesis_names = sorted(set(reference_names + ceiling_85_names + ceiling_90_names + avg_top2_names))
print(f"Jugadores en la tabla de síntesis ({len(synthesis_names)}):")
print(synthesis_names)

synthesis = role_scores_with_minutes[role_scores_with_minutes["name"].isin(synthesis_names)].merge(
    role_metrics_df[["id", "matches_played"]], on="id"
)

synthesis_display = synthesis[[
    "name", "minutesPlayed", "world_cups_played", "matches_played",
    "finisher_score", "dribbler_score", "chance_creator_score", "play_organizer_score",
    "role_score_avg", "n_elite_roles_90", "n_elite_roles_95", "n_elite_roles_99",
]].round(1).sort_values("role_score_avg", ascending=False)

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
print(synthesis_display.to_string(index=False))

Jugadores en la tabla de síntesis (15):
['Carlos Babington', 'Cristiano Ronaldo', 'Diego Armando Maradona', 'Ermindo Onega', 'Gheorghe Hagi', 'Heinz Flohe', 'Igor Chislenko', 'Johan Cruyff', 'Kylian Mbappé', 'Lionel Messi', 'Neymar', 'Rivellino', 'Robinho', 'Ronaldinho', 'Zinedine Zidane']
                  name  minutesPlayed  world_cups_played  matches_played  finisher_score  dribbler_score  chance_creator_score  play_organizer_score  role_score_avg  n_elite_roles_90  n_elite_roles_95  n_elite_roles_99
          Johan Cruyff          630.0                  1               7            89.4            98.7                  98.7                  86.5            93.3                 2                 2                 0
          Lionel Messi         3054.0                  6              34            97.2            95.7                  92.1                  84.1            92.3                 3                 2                 0
                Neymar         1225.0               

## Reading all the lenses together

No player in the 2488-person population clears the elite threshold (90th
percentile or higher) in all four roles at once. The strongest form of
the hypothesis, uniqueness across every dimension simultaneously, isn't
supported by this data, and that's reported as-is.

What the data does show, once every lens is read together rather than in
isolation:

**The ceiling that exists (3 of 4 roles, top 10%) is shared by 8 players,
and Messi is one of them, with by far the largest and longest sample of
the group.** The other seven who reach it (Neymar, Hagi, Onega,
Babington, Rivellino, Flohe, Chislenko, Robinho) each have 1-4 World Cups
and, except Neymar and Rivellino, well under 900 minutes of career
football. Messi reaches the same ceiling across 6 tournaments and 34
matches, a fundamentally different kind of evidence than a short hot
streak.

**By average role score, restricted to real career-length samples (900+
or 1500+ minutes), Messi has the single highest average in the
population.** Only at the loosest floor (270 minutes) does anyone edge
him out, Cruyff, whose entire World Cup career is a single tournament.
Small-sample players cluster near the top of the average-score ranking
in general (Onega, Babington, Chislenko, Flohe, Robinho all rank above
several multi-tournament players), which is expected: sustaining a high
rate over a handful of matches is a statistically easier event than
sustaining it across two decades, and this is exactly the pattern the
minutes-threshold sensitivity check was built to surface, not hide.

**Maradona is the closest large-sample rival, and a genuine one, not
cherry-picked.** Second by average score among 900+ and 1500+ minute
players (89.2), with his gap to Messi (92.3) explained almost entirely by
play organizer (75.1 vs. Messi's 84.1), the exact role predicted in Phase
4 to be Messi's relative weak point and the one furthest from the shared
skill cluster of the other three roles.

**The sharpest contrast is Cristiano Ronaldo**, whose career sample is
nearly identical to Messi's (2206 minutes, 6 World Cups, 27 matches) but
whose average role score is the lowest in this entire table (69.5),
elite in only 1 of 4 roles. Equal opportunity, comparable longevity, and
a specialized, one-dimensional statistical profile (96.6 as a finisher,
37.1 as an organizer). This is direct evidence that the finding isn't
simply "whoever plays the most World Cups wins": longevity alone doesn't
produce a high multi-role average, Cristiano had exactly as much of it as
Messi and it didn't.

---

## Leyendo todas las vistas juntas

Ningún jugador de la población de 2488 supera el umbral de élite
(percentil 90 o superior) en los cuatro roles a la vez. La forma más
fuerte de la hipótesis, ser único en las cuatro dimensiones
simultáneamente, no está respaldada por estos datos, y se reporta tal
cual.

Lo que los datos sí muestran, leyendo todas las vistas juntas en vez de
aisladas:

**El techo que existe (3 de 4 roles, top 10%) lo comparten 8 jugadores, y
Messi es uno de ellos, con por lejos la muestra más grande y sostenida
del grupo.** Los otros siete que lo alcanzan (Neymar, Hagi, Onega,
Babington, Rivellino, Flohe, Chislenko, Robinho) tienen entre 1 y 4
Mundiales cada uno y, salvo Neymar y Rivellino, muy por debajo de 900
minutos de carrera. Messi llega al mismo techo a lo largo de 6 torneos y
34 partidos, un tipo de evidencia fundamentalmente distinto a un pico
corto.

**Por promedio de puntaje de rol, restringido a muestras reales de
carrera (900+ o 1500+ minutos), Messi tiene el promedio más alto de toda
la población.** Solo en el piso más laxo (270 minutos) alguien lo supera,
Cruyff, cuya carrera mundialista completa es un solo torneo. Los
jugadores de muestra chica se agrupan cerca del top del ranking de
promedio en general (Onega, Babington, Chislenko, Flohe, Robinho quedan
todos por encima de varios jugadores multi-torneo), lo cual es esperable:
sostener una tasa alta en un puñado de partidos es un evento
estadísticamente más fácil que sostenerla a lo largo de dos décadas, y
es exactamente el patrón que el chequeo de sensibilidad al umbral de
minutos estaba diseñado para sacar a la luz, no para esconder.

**Maradona es el rival de muestra grande más cercano, y uno genuino, no
elegido a conveniencia.** Segundo por promedio entre jugadores de 900+ y
1500+ minutos (89.2), con su distancia a Messi (92.3) explicada casi por
completo por organizador de juego (75.1 contra el 84.1 de Messi), el
rol exacto que la Fase 4 predijo como el punto relativamente más débil de
Messi y el más alejado del grupo de habilidad compartido de los otros
tres.

**El contraste más marcado es Cristiano Ronaldo**, cuya muestra de
carrera es casi idéntica a la de Messi (2206 minutos, 6 Mundiales, 27
partidos) pero cuyo promedio de puntaje de rol es el más bajo de toda
esta tabla (69.5), élite en solo 1 de 4 roles. Misma oportunidad,
longevidad comparable, y un perfil estadístico especializado y
unidimensional (96.6 como finalizador, 37.1 como organizador). Esta es
evidencia directa de que el hallazgo no es simplemente "el que juega más
Mundiales gana": la longevidad sola no produce un promedio multi-rol
alto, Cristiano tuvo exactamente la misma y no le alcanzó.

In [12]:
role_scores_with_minutes.to_csv("../data/processed/role_scores.csv", index=False)
print(f"Guardado: {role_scores_with_minutes.shape[0]} jugadores, {role_scores_with_minutes.shape[1]} columnas")

Guardado: 2488 jugadores, 31 columnas


## Phase 5 conclusion

Tested the central question across multiple deliberately different
lenses instead of committing to one: binary elite-role count at three
percentile thresholds (90/95/99), a continuous average role score as a
complementary view, and sensitivity checks against three minimum-minutes
floors (270/900/1500) for both. No single lens was allowed to stand alone
as "the answer."

The strongest version of the hypothesis (elite in all 4 roles at once)
isn't supported: nobody in the population reaches that at a genuine elite
threshold. The version the data does support: among players with a real,
career-length World Cup sample, Messi has the highest average role score
in the population, reaches the same 3-of-4 ceiling as 7 other players
(all with far shorter samples), and the one role that keeps him short of
4-of-4, play organizer, is the exact role predicted in Phase 4 to be his
relative weak point before any of this was run. Maradona is a genuine,
non-cherry-picked runner-up by the same measure. Cristiano Ronaldo, with
a nearly identical career sample to Messi's, has the lowest average in
the comparison table, direct evidence the finding isn't just a function
of playing many World Cups.

**Result:** `data/processed/role_scores.csv` updated with elite-count
columns (3 thresholds) and `role_score_avg`. This, together with
`role_metrics.csv`, is the full evidentiary base for the project's
central claim. Phase 6 (role specialists) and the final dataset/Tableau
work build on top of this, not the other way around.

---

## Conclusión de la Fase 5

Se probó la pregunta central con varias vistas deliberadamente distintas
en vez de comprometerse con una sola: conteo binario de élite por rol en
tres umbrales de percentil (90/95/99), un promedio continuo de puntaje de
rol como vista complementaria, y chequeos de sensibilidad contra tres
pisos mínimos de minutos (270/900/1500) para ambas. Ninguna vista sola se
dejó parada como "la respuesta".

La versión más fuerte de la hipótesis (élite en los 4 roles a la vez) no
está respaldada: nadie en la población llega a eso en un umbral de élite
genuino. La versión que los datos sí respaldan: entre jugadores con una
muestra real de carrera mundialista, Messi tiene el promedio de puntaje
de rol más alto de la población, llega al mismo techo de 3-de-4 que otros
7 jugadores (todos con muestras mucho más cortas), y el único rol que lo
deja corto de 4-de-4, organizador de juego, es exactamente el rol que la
Fase 4 predijo como su punto relativamente más débil antes de correr
nada de esto. Maradona es un segundo lugar genuino, no elegido a
conveniencia, por la misma medida. Cristiano Ronaldo, con una muestra de
carrera casi idéntica a la de Messi, tiene el promedio más bajo de toda
la tabla de comparación, evidencia directa de que el hallazgo no es
simplemente una función de jugar muchos Mundiales.

**Resultado:** `data/processed/role_scores.csv` actualizado con las
columnas de conteo de élite (3 umbrales) y `role_score_avg`. Esto, junto
con `role_metrics.csv`, es la base de evidencia completa para la
afirmación central del proyecto. La Fase 6 (especialistas por rol) y el
dataset final/Tableau se construyen sobre esto, no al revés.
